In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np

data = pd.read_csv("weather_data.csv")

print(data.columns)# Convert DATE to actual datetime
data['DATE'] = pd.to_datetime(data['DATE'])

# Sort chronologically
data = data.sort_values('DATE')

# Remove columns we don't need
data = data.drop(columns=['STATION', 'NAME', 'SNWD'])

print(data.head())
data['TMAX'] = (data['TMAX'] - 32) * 5/9
data['TMIN'] = (data['TMIN'] - 32) * 5/9
print(data.head())
print(data.describe())
features = data[['PRCP', 'TMAX', 'TMIN']].values
X = []
y = []

sequence_length = 14

for i in range(len(features) - sequence_length):

    X.append(
        features[i:i + sequence_length]
    )

    y.append(
        features[i + sequence_length, 1]
    )
print(X[0])
print(y[0])

X = np.array(X)
y = np.array(y)

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

y = y.reshape(-1, 1)

print(X.shape)
print(y.shape)
train_size = int(0.70 * len(X))

val_size = int(0.15 * len(X))

X_train = X[:train_size]
y_train = y[:train_size]

X_val = X[train_size:train_size + val_size]
y_val = y[train_size:train_size + val_size]

X_test = X[train_size + val_size:]
y_test = y[train_size + val_size:]

print("Training:")
print(X_train.shape)
print(y_train.shape)

print("Validation:")
print(X_val.shape)
print(y_val.shape)

print("Testing:")
print(X_test.shape)
print(y_test.shape)

X_mean = X_train.mean(dim=(0, 1), keepdim=True)
X_std = X_train.std(dim=(0, 1), keepdim=True)

X_train = (X_train - X_mean) / X_std
X_val = (X_val - X_mean) / X_std
X_test = (X_test - X_mean) / X_std

y_mean = y_train.mean()
y_std = y_train.std()

y_train = (y_train - y_mean) / y_std
y_val = (y_val - y_mean) / y_std
y_test = (y_test - y_mean) / y_std

from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

trainloader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

valloader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

testloader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

Index(['STATION', 'NAME', 'DATE', 'PRCP', 'SNWD', 'TMAX', 'TMIN'], dtype='str')
        DATE  PRCP  TMAX  TMIN
0 1985-11-01  0.00    45    37
1 1985-11-02  0.00    47    38
2 1985-11-03  0.00    44    32
3 1985-11-04  0.11    40    28
4 1985-11-05  0.07    61    38
        DATE  PRCP       TMAX      TMIN
0 1985-11-01  0.00   7.222222  2.777778
1 1985-11-02  0.00   8.333333  3.333333
2 1985-11-03  0.00   6.666667  0.000000
3 1985-11-04  0.11   4.444444 -2.222222
4 1985-11-05  0.07  16.111111  3.333333
                             DATE          PRCP          TMAX          TMIN
count                       14791  14791.000000  14791.000000  14791.000000
mean   2006-03-24 04:37:16.278818      0.070909     15.624178      7.667613
min           1985-11-01 00:00:00      0.000000    -11.111111    -15.000000
25%           1995-12-16 12:00:00      0.000000      8.888889      2.777778
50%           2006-04-01 00:00:00      0.000000     15.555556      7.777778
75%           2016-06-15 12:00:00     

In [4]:
class WeatherLSTM(nn.Module):
    def __init__(self):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=3,
            hidden_size=64,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Linear(64, 1)

    def forward(self, x):

        output, (hidden, cell) = self.lstm(x)

        x = hidden[-1]

        x = self.fc(x)

        return x
lstm_model = WeatherLSTM()

print(lstm_model)   
criterion = nn.MSELoss()

lstm_optimizer = torch.optim.Adam(
    lstm_model.parameters(),
    lr=0.001
)
epochs = 50

for epoch in range(epochs):

    # TRAINING
    lstm_model.train()

    train_loss = 0.0

    for X_batch, y_batch in trainloader:

        predictions = lstm_model(X_batch)

        loss = criterion(predictions, y_batch)

        lstm_optimizer.zero_grad()

        loss.backward()

        lstm_optimizer.step()

        train_loss += loss.item()

    train_loss /= len(trainloader)


    # VALIDATION
    lstm_model.eval()

    val_loss = 0.0

    with torch.no_grad():

        for X_batch, y_batch in valloader:

            predictions = lstm_model(X_batch)

            loss = criterion(predictions, y_batch)

            val_loss += loss.item()

    val_loss /= len(valloader)


    print(
        f"Epoch {epoch+1}/{epochs} "
        f"Train Loss: {train_loss:.4f} "
        f"Validation Loss: {val_loss:.4f}"
    )
    lstm_model.eval()

lstm_predictions = []
lstm_actual = []

with torch.no_grad():

    for X_batch, y_batch in testloader:

        predictions = lstm_model(X_batch)

        lstm_predictions.append(predictions)
        lstm_actual.append(y_batch)
lstm_predictions = torch.cat(lstm_predictions)
lstm_actual = torch.cat(lstm_actual)
lstm_predictions_real = (
    lstm_predictions * y_std + y_mean
)

lstm_actual_real = (
    lstm_actual * y_std + y_mean
)
lstm_mae = torch.mean(
    torch.abs(
        lstm_predictions_real - lstm_actual_real
    )
)

lstm_rmse = torch.sqrt(
    torch.mean(
        (lstm_predictions_real - lstm_actual_real) ** 2
    )
)

print("LSTM MAE:", lstm_mae.item())
print("LSTM RMSE:", lstm_rmse.item())

WeatherLSTM(
  (lstm): LSTM(3, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)
Epoch 1/50 Train Loss: 0.2004 Validation Loss: 0.1352
Epoch 2/50 Train Loss: 0.1060 Validation Loss: 0.1094
Epoch 3/50 Train Loss: 0.0970 Validation Loss: 0.1063
Epoch 4/50 Train Loss: 0.0955 Validation Loss: 0.1053
Epoch 5/50 Train Loss: 0.0950 Validation Loss: 0.1044
Epoch 6/50 Train Loss: 0.0942 Validation Loss: 0.1084
Epoch 7/50 Train Loss: 0.0931 Validation Loss: 0.1042
Epoch 8/50 Train Loss: 0.0929 Validation Loss: 0.1052
Epoch 9/50 Train Loss: 0.0926 Validation Loss: 0.1028
Epoch 10/50 Train Loss: 0.0914 Validation Loss: 0.1035
Epoch 11/50 Train Loss: 0.0919 Validation Loss: 0.1028
Epoch 12/50 Train Loss: 0.0909 Validation Loss: 0.1021
Epoch 13/50 Train Loss: 0.0908 Validation Loss: 0.1023
Epoch 14/50 Train Loss: 0.0911 Validation Loss: 0.1023
Epoch 15/50 Train Loss: 0.0905 Validation Loss: 0.1054
Epoch 16/50 Train Loss: 0.0899 Validation Loss: 0.1032
Epoch 17/50 Trai